<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_36_drf_fastapi/note_lesson_36_drf_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔌 Урок 36 — DRF: API до нотаток як рефакторинг

Застосунок нотаток з уроку 35 уміє все для людей: сторінки, форми, вхід. Сьогодні нотатки отримують **програми** — бот, Streamlit, мобільний застосунок: додаємо REST API на Django REST Framework (DRF) поверх тих самих `services` і `selectors`, не чіпаючи ні моделей, ні сторінок. Наприкінці — той самий API на FastAPI.

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | `crispy_notes_project` з API у ноутбуці, свіжі дані `seed()` | 0.1 | |
| 1 | серіалізатори: вихід, `many=True`, JSON, `source=`, вхід і помилки, значення за замовчуванням і `partial`, приведення типів, чужий записник, `SerializerMethodField`, `validate_<поле>()`, небезпека `"__all__"` | 1.1–1.10 | **Задача 1** |
| 2 | API через `APIClient`: роутер, анонім, Basic і сесія, `force_authenticate`, створення з «підкинутим» власником, `400`, `retrieve` / `pin` / `DELETE`, `405` | 2.1–2.8 | |
| 3 | IDOR і `PATCH`: чужа нотатка — `404`, часткова зміна, фільтри, баг `updated_at`, `ModelViewSet` віддає чужі нотатки | 3.1–3.5 | **Задача 2** |
| 4 | OpenAPI-схема | 4.1 | |
| 5 | той самий API на FastAPI: CRUD, `422` проти `400`, `PATCH` з `exclude_unset`, схема, свій мінізастосунок | 5.1–5.5 | **Задача 3** |

**Як працювати**

- Виконуй клітинки **зверху вниз**. Клітинка з визначенням (`def`, класи, константи) лише створює їх — **запуск завжди в наступній клітинці**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- Рядки `Forbidden: …`, `Bad Request: …`, `Not Found: …` у виводі — журнал Django, не `print`: кожна відповідь 4xx потрапляє в журнал.
- Теорія, діфи рефакторингу й архітектура — у книзі курсу: [Урок 36](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_36/); поглиблено — глава Django-книги [REST API: Django REST Framework](https://nikoriakviktor.github.io/notes_chat_app/06_application_architecture/drf_rest_api_full/).

---
## 0. Проєкт у ноутбуці

Проєкт — `crispy_notes_project` з уроку 35 + DRF: `hello_app/api.py` (серіалізатори й `NoteViewSet` поверх `services`/`selectors`) і роутер у `hello_project/urls.py`. Поруч лежить `fastapi_notes.py` — той самий API на FastAPI. У Colab клітинка нижче завантажить лише папку уроку і встановить залежності з `requirements.txt` проєкту.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_36_drf_fastapi"
REPO = "https://github.com/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026.git"

if os.path.basename(os.getcwd()) == "crispy_notes_project":   # клітинку запустили вдруге
    os.chdir("..")
if not os.path.isdir("crispy_notes_project"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import crispy_bootstrap5, debug_toolbar, drf_spectacular, fastapi, httpx, rest_framework  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "crispy_notes_project/requirements.txt",
                    "fastapi", "httpx"], check=True)

sys.path.insert(0, os.getcwd())                        # fastapi_notes.py лежить у папці уроку
os.chdir("crispy_notes_project")
result = subprocess.run([sys.executable, "manage.py", "migrate"], capture_output=True, text=True)
print("\n".join((result.stdout or result.stderr).splitlines()[-2:]))

In [ ]:
import base64
import json
import time
from types import SimpleNamespace

import django

sys.path.insert(0, os.getcwd())
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hello_project.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"      # Jupyter працює в asyncio-циклі (урок 34)
django.setup()

from django.contrib.auth.models import User
from django.test.utils import setup_test_environment
from rest_framework import permissions, serializers, viewsets
from rest_framework.renderers import JSONRenderer
from rest_framework.test import APIClient, APIRequestFactory, force_authenticate

from hello_app import selectors, services
from hello_app.api import NoteInputSerializer, NoteOutputSerializer
from hello_app.models import Note, Notebook

setup_test_environment()                               # хост testserver для тестового клієнта
print("Django", django.get_version(), "| проєкт:", os.path.basename(os.getcwd()))

### Приклад 0.1. Свіжі дані одним викликом

Двоє користувачів — Олена і Боб — з нотатками й записниками, створені **через ті самі `services`**, що й на сторінках. `seed()` спершу видаляє обох (разом з усім, що їм належить), тож її можна викликати на початку кожного розділу.

In [ ]:
def seed():
    """Олена: записник «Навчання» і 2 нотатки; Боб: свій записник і 1 нотатка."""
    User.objects.filter(username__in=["olena", "bob"]).delete()
    olena = User.objects.create_user("olena", password="pass-12345")
    bob = User.objects.create_user("bob", password="pass-12345")
    study = Notebook.objects.create(user=olena, title="Навчання")
    bob_notebook = Notebook.objects.create(user=bob, title="Записник Боба")
    drf_note = services.create_note(user=olena, title="Вивчити DRF", content="serializers, viewsets",
                                    priority=3, notebook=study)
    services.create_note(user=olena, title="Купити квитки", priority=2)
    bob_note = services.create_note(user=bob, title="Нотатка Боба")
    return olena, bob, study, bob_notebook, drf_note, bob_note

In [ ]:
olena, bob, study, bob_notebook, drf_note, bob_note = seed()
print("нотаток: Олена", olena.notes.count(), "| Боб", bob.notes.count())

---
## 1. Серіалізатори

**Серіалізатор** для API — те саме, що форма для сторінки (урок 35): перетворює дані й перевіряє їх, лише замість HTML — JSON.

```
вихід:  Note (об'єкт моделі)  → NoteOutputSerializer(note).data → dict → JSON у відповідь
вхід:   JSON з тіла запиту    → NoteInputSerializer(data=...)   → is_valid() → validated_data або errors
```

У проєкті їх **два**: клієнт бачить більше, ніж може змінити (`id`, `priority_label`, назву записника, `updated_at`), а надіслати може лише дозволене.

### Приклад 1.1. Вихідний серіалізатор: одна нотатка

In [ ]:
data = NoteOutputSerializer(drf_note).data

assert data["title"] == "Вивчити DRF" and data["priority"] == 3
assert data["priority_label"] == "🟠 Високий" and data["notebook"] == "Навчання"
assert "user" not in data, "власника клієнт не бачить"
print({key: value for key, value in data.items() if key != "updated_at"})
print("updated_at:", data["updated_at"][:19], "…", type(data["updated_at"]).__name__)

Поля — рівно ті, що в `Meta.fields`, у тому ж порядку; `user` немає. `updated_at` прийшов **рядком** у форматі ISO 8601: JSON не має типу «дата», тож серіалізатор сам перетворив `datetime` на текст.

### Приклад 1.2. Список: `many=True` і справжній JSON

Для списку — `many=True`. А в тіло відповіді потрапляють **байти** JSON: їх робить рендерер (`JSONRenderer`) з того самого словника.

In [ ]:
notes = selectors.get_user_notes(olena)
listing = NoteOutputSerializer(notes, many=True).data
print(type(listing).__name__, len(listing), [note["title"] for note in listing])

body = JSONRenderer().render([{"id": n["id"], "title": n["title"]} for n in listing])
print(type(body).__name__, body.decode())

`ReturnList` — список з посиланням на серіалізатор (для помилок і Browsable API). Порядок — з `selectors.get_user_notes`: спершу закріплені, потім вищий пріоритет. Кирилиця в JSON — як є, не `В…`: DRF за замовчуванням пише UTF-8.

### Приклад 1.3. `source=`: поле з методу чи зв'язаного об'єкта

`priority_label` — це `source="get_priority_display"` (метод моделі для `choices`), `notebook` — `source="notebook.title"`. Що буде, коли записника немає?

In [ ]:
tickets = Note.objects.get(user=olena, title="Купити квитки")
for note in (drf_note, tickets):
    out = NoteOutputSerializer(note).data
    print(f"{out['title']:14} priority={out['priority']} → {out['priority_label']!r:14} notebook={out['notebook']!r}")

`source` може бути методом (його викликають) або шляхом через крапку. Для «Купити квитки» `note.notebook` — `None`, і `notebook.title` впав би з `AttributeError`; `default=None` у полі каже «немає — поверни `None`».

### Приклад 1.4. Вхідний серіалізатор: помилки за полями

In [ ]:
bad = NoteInputSerializer(data={"title": "", "priority": 9})
bad.is_valid()
bad_fields = set(bad.errors)

assert bad_fields == {"title", "priority"}
for field, errors in bad.errors.items():
    print(field, "→", errors[0], "| код:", errors[0].code)

Як у форми: `is_valid()` → `False`, `errors` — словник «поле → список помилок». Кожна помилка — `ErrorDetail`: рядок з кодом (`blank`, `invalid_choice`), щоб тести й клієнти перевіряли код, а не текст.

**🔮 Прогноз:** Що буде в `validated_data` для `{'title': 'План'}` — лише `title`?

<details>
<summary>Відповідь</summary>

Ні: серіалізатор підставить значення за замовчуванням — `content=''`, `priority=1`, `is_pinned=False`, `notebook=None`. Інакше буде з `partial=True` (для `PATCH`): там у `validated_data` лише надіслані поля.

</details>

### Приклад 1.5. Перевіряємо прогноз: `default` і `partial=True`

In [ ]:
full = NoteInputSerializer(data={"title": "План"})
part = NoteInputSerializer(data={"title": "План"}, partial=True)
print("POST :", full.is_valid(), dict(full.validated_data))
print("PATCH:", part.is_valid(), dict(part.validated_data))
print("PATCH без title:", NoteInputSerializer(data={}, partial=True).is_valid())

Саме тому `partial_update` у ViewSet передає `partial=True`: `PATCH {"title": ...}` не має скидати пріоритет до `1`. А обов'язковий `title` при `partial=True` теж не обов'язковий — порожній `PATCH` валідний і нічого не змінює.

### Приклад 1.6. Приведення типів

Клієнти бувають різні: один надішле `priority: 3`, інший — `"3"`, форма в браузері — `is_pinned: "true"`. Серіалізатор приводить значення до потрібного типу — або повертає помилку.

In [ ]:
for raw in ({"title": "A", "priority": "3", "is_pinned": "true"},
            {"title": "B", "priority": 3, "is_pinned": 1},
            {"title": "C", "is_pinned": "так"},
            {"title": 42}):
    s = NoteInputSerializer(data=raw)
    print(raw, "→", s.is_valid() and {k: s.validated_data[k] for k in ("title", "priority", "is_pinned")} or dict(s.errors))

- `"3"` → `3`, `"true"` і `1` → `True`: DRF приймає поширені записи;
- `"так"` — помилка: `BooleanField` знає лише англійські `true/false`, `yes/no`, `on/off`, `1/0`;
- `title: 42` → рядок `"42"`: `CharField` перетворює числа на текст. Якщо це небажано, є `serializers.CharField(...)` з власною перевіркою (приклад 1.9).

### Приклад 1.7. Чужий записник — помилка валідації

Вхідний серіалізатор фільтрує записники за користувачем з `context["request"]` — як `NoteForm(user=...)` в уроці 35. Справжнього запиту тут немає, тож підставимо об'єкт з полем `user`.

In [ ]:
as_olena = {"request": SimpleNamespace(user=olena)}
serializer = NoteInputSerializer(data={"title": "Чужий записник", "notebook": bob_notebook.pk}, context=as_olena)
ok = serializer.is_valid()

assert ok is False and "notebook" in serializer.errors
own = NoteInputSerializer(data={"title": "Свій записник", "notebook": study.pk}, context=as_olena)
assert own.is_valid() and own.validated_data["notebook"] == study
print("чужий:", serializer.errors["notebook"][0])
print("свій:", own.validated_data["notebook"])

`PrimaryKeyRelatedField` шукає `id` **у своєму `queryset`** — а там лише записники Олени. Для неї записник Боба «не існує»; у `validated_data` для свого — уже об'єкт `Notebook`, а не число.

### Приклад 1.8. Обчислене поле: `SerializerMethodField`

Клієнту потрібне поле, якого немає в моделі, — наприклад «чи довга нотатка» і короткий уривок тексту. `SerializerMethodField` викликає метод `get_<поле>(self, obj)`.

In [ ]:
class NotePreviewSerializer(serializers.ModelSerializer):
    preview = serializers.SerializerMethodField()
    is_long = serializers.SerializerMethodField()

    class Meta:
        model = Note
        fields = ["id", "title", "preview", "is_long"]

    def get_preview(self, note):
        return note.content[:12] + ("…" if len(note.content) > 12 else "")

    def get_is_long(self, note):
        return len(note.content) > 12

In [ ]:
for item in NotePreviewSerializer(selectors.get_user_notes(olena), many=True).data:
    print(dict(item))

Поле лише для читання: надіслати `preview` клієнт не може. Метод отримує **об'єкт моделі** — можна рахувати що завгодно, але кожен запит до бази в такому методі повториться для кожної нотатки списку (проблема N+1, урок 39).

### Приклад 1.9. Власні перевірки: `validate_<поле>()` і `validate()`

Ті самі ідеї, що `clean_<поле>()` і `clean()` у формах (урок 35): одне поле — метод `validate_title`, кілька полів — `validate(attrs)`. Помилка — `serializers.ValidationError`.

In [ ]:
class StrictNoteInputSerializer(NoteInputSerializer):
    def validate_title(self, title):
        if title.isdigit():
            raise serializers.ValidationError("Заголовок не може складатися лише з цифр.")
        return " ".join(title.split())

    def validate(self, attrs):
        if attrs.get("is_pinned") and attrs.get("priority") == Note.PRIORITY_LOW:
            raise serializers.ValidationError({"is_pinned": "Закріплювати варто лише важливі нотатки."})
        return attrs

In [ ]:
for raw in ({"title": "2026"}, {"title": "Купити   квитки", "is_pinned": True},
            {"title": "Купити   квитки", "priority": 4, "is_pinned": True}):
    s = StrictNoteInputSerializer(data=raw)
    print(raw, "→", s.validated_data["title"] if s.is_valid() else {k: [str(e) for e in v] for k, v in s.errors.items()})

Успадкування дало все з `NoteInputSerializer` (поля, фільтр записників), а нові методи додали правила. Словник у `ValidationError` прив'язує помилку з `validate()` до конкретного поля — клієнт побачить її під `is_pinned`, а не в загальному `non_field_errors`.

### Приклад 1.10. Чому не `fields = "__all__"`

Найкоротший шлях — один `ModelSerializer` з усіма полями. Подивимось, що він відкриває.

In [ ]:
class AllFieldsSerializer(serializers.ModelSerializer):
    class Meta:
        model = Note
        fields = "__all__"

In [ ]:
print("поля:", list(AllFieldsSerializer().fields))
lazy = AllFieldsSerializer(data={"title": "Підкинута", "user": bob.pk, "is_archived": True})
print("валідний:", lazy.is_valid(), "| user у validated_data:", lazy.validated_data.get("user"),
      "| is_archived:", lazy.validated_data.get("is_archived"))

`"__all__"` показує `user` (чий це запис) і, гірше, **приймає** його на вході: Олена могла б створити нотатку від імені Боба, а заразом архівувати чи пересунути що завгодно. Тому в проєкті — явні списки полів, а `user` немає ні в вході, ні у виході: власника задає сервер з `request.user`.

### Задача 1. Картка нотатки для бота

Telegram-боту (урок 48) потрібна коротка картка нотатки. Напиши `NoteCardSerializer(serializers.ModelSerializer)`:

- `fields = ["id", "title", "priority_label", "notebook", "words"]`;
- `priority_label` і `notebook` — як у `NoteOutputSerializer` (приклади 1.1 і 1.3: `source=…`, `read_only=True`, для записника `default=None`);
- `words` — `SerializerMethodField`: кількість слів у `content` (`len(note.content.split())`).

Перевірка — у наступній клітинці.

In [ ]:
class NoteCardSerializer(serializers.ModelSerializer):
    # YOUR CODE HERE
    pass


In [ ]:
olena, bob, study, bob_notebook, drf_note, bob_note = seed()
assert hasattr(NoteCardSerializer, "Meta"), "опиши class Meta з model = Note і списком fields"
card = dict(NoteCardSerializer(drf_note).data)
assert list(card) == ["id", "title", "priority_label", "notebook", "words"], f"поля картки: {list(card)}"
assert card == {"id": drf_note.pk, "title": "Вивчити DRF", "priority_label": "🟠 Високий",
                "notebook": "Навчання", "words": 2}, card
tickets = Note.objects.get(user=olena, title="Купити квитки")
assert NoteCardSerializer(tickets).data["notebook"] is None, "без записника — None, а не помилка"
assert NoteCardSerializer(tickets).data["words"] == 0
assert len(NoteCardSerializer(selectors.get_user_notes(olena), many=True).data) == 2
print(card)
print("✅ Задача 1 розв'язана")

---
## 2. API через `APIClient`

`APIClient` — тестовий клієнт DRF: надсилає запити без сервера, `format="json"` — тіло як JSON, `force_authenticate(user)` — «увійти» без пароля (лише для тестів).

```
GET    /api/notes/            list            selectors.get_user_notes     200
POST   /api/notes/            create          services.create_note         201
GET    /api/notes/{id}/       retrieve        selectors.get_note_detail    200
PATCH  /api/notes/{id}/       partial_update  services.update_note         200
DELETE /api/notes/{id}/       destroy         services.delete_note         204
POST   /api/notes/{id}/pin/   pin (@action)   services.toggle_pin_note     200
```

### Приклад 2.1. Які адреси створив роутер

`DefaultRouter` перетворює методи `NoteViewSet` на маршрути. Подивимось на них — без службових варіантів з розширенням формату (`.json`).

In [ ]:
from hello_project.urls import router

for url in router.urls:
    if "format" not in str(url.pattern):
        print(f"{str(url.pattern):40} name={url.name}")

Чотири маршрути з одного класу — це регулярні вирази відносно `/api/`: список (`note-list`), один об'єкт (`note-detail`, `pk` — будь-що без `/` і `.`), власна дія `pin` (`note-pin`, з `@action(detail=True)`) і корінь API (`api-root`, порожній шлях — сама адреса `/api/`) — сторінка зі списком ресурсів. Імена — з `basename="note"`; за ними працює `reverse("note-detail", args=[5])`.

### Приклад 2.2. Анонім — `403`

In [ ]:
olena, bob, study, bob_notebook, drf_note, bob_note = seed()
anon = APIClient()
response = anon.get("/api/notes/")
anon_code = response.status_code

assert anon_code == 403, "IsAuthenticated; перший клас автентифікації — сесійний, тому 403, а не 401"
print(anon_code, response.data, "| WWW-Authenticate:", response.get("WWW-Authenticate"))

`IsAuthenticated` у `REST_FRAMEWORK` закрив API для всіх без входу. Чому `403`, а не `401`: `401` вимагає заголовка `WWW-Authenticate` («як саме увійти»), а його бере DRF з **першого** класу автентифікації — `SessionAuthentication`, який такого заголовка не має.

### Приклад 2.3. Два способи увійти: Basic і сесія

`BasicAuthentication` — логін і пароль у заголовку `Authorization` (для `curl` і скриптів). `SessionAuthentication` — cookie сесії після входу, той самий, що на сторінках сайту.

In [ ]:
def basic(username, password):
    """Заголовок Authorization для Basic: 'Basic ' + base64('логін:пароль')."""
    token = base64.b64encode(f"{username}:{password}".encode()).decode()
    return {"HTTP_AUTHORIZATION": f"Basic {token}"}

In [ ]:
script = APIClient()
script.credentials(**basic("olena", "pass-12345"))
print("Basic, правильний пароль →", script.get("/api/notes/").status_code)
script.credentials(**basic("olena", "wrong"))
wrong = script.get("/api/notes/")
print("Basic, хибний пароль     →", wrong.status_code, wrong.data["detail"])

browser = APIClient()
print("сесія: вхід              →", browser.login(username="olena", password="pass-12345"))
print("сесія: список            →", browser.get("/api/notes/").status_code, "| cookie:", sorted(browser.cookies))

- з правильним паролем — `200`; з хибним — відмова з поясненням (код знову `403`, бо заголовок `WWW-Authenticate` бере DRF з першого класу автентифікації — сесійного);
- `login()` поставив cookie `sessionid` — і API впізнав Олену так само, як сторінки. Пароль у кожному запиті — лише для скриптів через HTTPS; токени для мобільних застосунків — урок 41.

### Приклад 2.4. Список: лише свої нотатки

In [ ]:
api = APIClient()
api.force_authenticate(olena)
titles = [note["title"] for note in api.get("/api/notes/").json()]

assert titles == ["Вивчити DRF", "Купити квитки"], "лише свої; спершу вищий пріоритет"
print(titles)

Нотатки Боба немає: `list` викликає `selectors.get_user_notes(request.user)`. Порядок — з selector: пріоритет 3, потім 2.

### Приклад 2.5. Створення з «підкинутим» власником

In [ ]:
created = api.post("/api/notes/", {"title": "Здати проєкт", "priority": 4, "user": bob.pk}, format="json")

assert created.status_code == 201
note = Note.objects.get(pk=created.json()["id"])
assert note.user == olena, "власника задає сервер (request.user), поле user ігнорується"
print(created.status_code, created.json()["priority_label"], "| власник:", note.user)

`201 Created` — ресурс створено. `"user": bob.pk` проігноровано мовчки: такого поля у вхідному серіалізаторі немає, а `create` передає в сервіс `user=request.user`.

### Приклад 2.6. Помилки валідації по HTTP — `400`

In [ ]:
bad = api.post("/api/notes/", {"title": "", "priority": 9}, format="json")
print(bad.status_code, bad["Content-Type"])
print(json.dumps(bad.json(), ensure_ascii=False))

`raise_exception=True` у `_input()` перетворив помилки серіалізатора на відповідь `400 Bad Request` — той самий словник «поле → помилки», уже JSON. Клієнт знає, що саме виправити.

### Приклад 2.7. Одна нотатка, власна дія `pin`, видалення

In [ ]:
note_id = created.json()["id"]
one = api.get(f"/api/notes/{note_id}/")
print("GET    →", one.status_code, one.json()["title"], "| закріплена:", one.json()["is_pinned"])
pinned = api.post(f"/api/notes/{note_id}/pin/")
print("pin    →", pinned.status_code, pinned.json()["is_pinned"])
print("список →", [n["title"] for n in api.get("/api/notes/").json()][:1], "першою")
gone = api.delete(f"/api/notes/{note_id}/")
print("DELETE →", gone.status_code, repr(gone.content), "| у базі:", Note.objects.filter(pk=note_id).exists())

Закріплена нотатка піднялася на перше місце списку — порядок задає selector. `204 No Content` — успіх без тіла: видаленому ресурсу нічого показувати.

### Приклад 2.8. Метод, якого немає, — `405`

`NoteViewSet` навмисно не має `update` (повна заміна `PUT`) — лише `partial_update` (`PATCH`).

In [ ]:
put = api.put(f"/api/notes/{drf_note.pk}/", {"title": "Повна заміна"}, format="json")
print(put.status_code, put.data["detail"])
print("Allow:", put["Allow"])

`405 Method Not Allowed`, а заголовок `Allow` перелічує, що можна з цією адресою. Методи ViewSet — це і є список дозволених дій.

---
## 3. IDOR і `PATCH`

**IDOR** (Insecure Direct Object Reference) — отримати чужий об'єкт, підставивши його `id` в адресу. Захист у проєкті — `selectors.get_note_detail(request.user, pk)`: нотатку шукають **серед нотаток користувача**.

### Приклад 3.1. Чужа нотатка — `404`

In [ ]:
olena, bob, study, bob_notebook, drf_note, bob_note = seed()
api = APIClient()
api.force_authenticate(olena)
read_code = api.get(f"/api/notes/{bob_note.pk}/").status_code
patch_code = api.patch(f"/api/notes/{bob_note.pk}/", {"title": "Зламано"}, format="json").status_code
delete_code = api.delete(f"/api/notes/{bob_note.pk}/").status_code

assert (read_code, patch_code, delete_code) == (404, 404, 404), "selectors.get_note_detail шукає серед нотаток користувача"
assert Note.objects.get(pk=bob_note.pk).title == "Нотатка Боба"
print(read_code, patch_code, delete_code)

**🔮 Прогноз:** Чому чужа нотатка — `404`, а не `403 Forbidden`?

<details>
<summary>Відповідь</summary>

`403` підтвердив би, що нотатка з таким `id` існує. `404` не розкриває нічого — перебирати чужі `id` марно.

</details>

### Приклад 3.2. `PATCH`: змінюється лише надіслане

In [ ]:
note = Note.objects.create(user=olena, title="Здати проєкт", priority=4)
before = note.updated_at
time.sleep(0.01)
patched = api.patch(f"/api/notes/{note.pk}/", {"title": "Здати проєкт до п'ятниці"}, format="json")
note.refresh_from_db()

assert patched.status_code == 200
assert (note.title, note.priority) == ("Здати проєкт до п'ятниці", 4), "PATCH не чіпає ненадіслані поля"
assert note.updated_at > before, "update_note зберігає й updated_at"
print(patched.status_code, note.title, note.priority, "| updated_at зріс на", note.updated_at - before)

`priority` лишився `4`: `partial=True` не підставив значення за замовчуванням (приклад 1.5), а `services.update_note` змінює лише поля, що прийшли. `updated_at` оновився — чому це не дрібниця, у прикладі 3.4.

### Приклад 3.3. Фільтри списку: `?notebook=` і `?search=`

`list` читає параметри запиту й передає їх у `selectors.get_user_notes` — нового ORM-запиту у view немає. Записник шукаємо **серед своїх**.

In [ ]:
for query in ({"notebook": study.pk}, {"notebook": bob_notebook.pk}, {"notebook": "abc"}, {"search": "квитки"}):
    r = api.get("/api/notes/", query)
    print(query, "→", r.status_code, [n["title"] for n in r.data] if r.status_code == 200 else r.data["detail"])

- свій записник — лише його нотатки; чужий — `404`, як чужа нотатка;
- `"abc"` не падає з `500`: `isdigit()` відсікає нечислове значення до запиту в базу;
- `search` уже вмів selector — у API він з'явився одним рядком.

### Приклад 3.4. Знайди помилку зі сторінки уроку: `update_fields` і `auto_now`

API повертає `updated_at`. Подивимось, як воно оновлюється, коли модель зберігають з `update_fields`.

In [ ]:
note = Note.objects.create(user=olena, title="Стара назва")
before = note.updated_at
time.sleep(0.01)

note.title = "Нова назва"
note.save(update_fields=["title"])                 # зберегти лише title
note.refresh_from_db()
print("лише title         → змінився updated_at?", note.updated_at != before)

note.title = "Ще новіша назва"
note.save(update_fields=["title", "updated_at"])   # title разом з updated_at
note.refresh_from_db()
print("title + updated_at → змінився updated_at?", note.updated_at != before)

`auto_now=True` виставляє час у `pre_save()`, але Django записує в базу **лише поля з `update_fields`**. Якби `services.update_note` не додавав `updated_at`, час зміни назавжди лишався б часом створення, а відредагована нотатка не піднімалася б у списку (selector сортує за `-updated_at`). Тиха помилка: жодного винятку. У проєкті вже правильно — `update_fields=changed_fields + ['updated_at']`, тому приклад 3.2 пройшов.

### Приклад 3.5. «Спрощений» API: `ModelViewSet` без selectors

Колега замінив серіалізатори й services одним класом:

```python
class QuickNoteViewSet(viewsets.ModelViewSet):
    queryset = Note.objects.all()
    serializer_class = NoteOutputSerializer
    permission_classes = [permissions.IsAuthenticated]
```

Вхід є, права є. Перевіримо, що бачить Олена. `titles_for` викликає `list` будь-якого ViewSet напряму — через `APIRequestFactory`, без роутера.

In [ ]:
class QuickNoteViewSet(viewsets.ModelViewSet):
    queryset = Note.objects.all()
    serializer_class = NoteOutputSerializer
    permission_classes = [permissions.IsAuthenticated]


def titles_for(viewset, user):
    """Заголовки, які `list` цього ViewSet повертає користувачу."""
    request = APIRequestFactory().get("/notes/")
    force_authenticate(request, user=user)
    return sorted(note["title"] for note in viewset.as_view({"get": "list"})(request).data)

In [ ]:
olena, bob, study, bob_notebook, drf_note, bob_note = seed()
leak = titles_for(QuickNoteViewSet, olena)
assert "Нотатка Боба" in leak, "помилка відтворена: чужа нотатка у списку"
print("Олена бачить:", leak)

Олена бачить **усі** нотатки, разом з нотаткою Боба. `IsAuthenticated` перевіряє лише «чи увійшов», а не «чиє це». `ModelViewSet` бере `queryset = Note.objects.all()` для кожного запиту — і для `retrieve`, `update`, `destroy` теж: чужу нотатку можна ще й змінити чи видалити.

### Задача 2. Виправ: лише свої нотатки

Допиши `SafeNoteViewSet.get_queryset()`: повертай нотатки лише поточного користувача — через `selectors.get_user_notes(self.request.user)`. `get_queryset()` ViewSet викликає для **кожної** дії, тож виправлення закриє і список, і доступ за `id`.

Перевірка — у наступній клітинці.

In [ ]:
class SafeNoteViewSet(QuickNoteViewSet):
    def get_queryset(self):
        # YOUR CODE HERE
        pass


In [ ]:
olena, bob, study, bob_notebook, drf_note, bob_note = seed()
view = SafeNoteViewSet()
view.request = SimpleNamespace(user=olena)
assert view.get_queryset() is not None, "get_queryset() має повернути нотатки поточного користувача (QuerySet)"
safe = titles_for(SafeNoteViewSet, olena)
assert "Нотатка Боба" not in safe, f"Олена все ще бачить чуже: {safe}"
assert safe == ["Вивчити DRF", "Купити квитки"]
assert titles_for(SafeNoteViewSet, bob) == ["Нотатка Боба"]
request = APIRequestFactory().get(f"/notes/{bob_note.pk}/")
force_authenticate(request, user=olena)
assert SafeNoteViewSet.as_view({"get": "retrieve"})(request, pk=bob_note.pk).status_code == 404, "чужа за id — 404"
print("було:", titles_for(QuickNoteViewSet, olena), "→ стало:", safe)
print("✅ Задача 2 розв'язана")

---
## 4. OpenAPI-схема

drf-spectacular будує опис API за стандартом OpenAPI з ViewSet і серіалізаторів — те, що FastAPI робить сам. Зі схеми Swagger UI малює інтерактивну документацію, а Postman імпортує всі запити (урок 38).

### Приклад 4.1. `/api/schema/`

In [ ]:
r = api.get("/api/schema/", {"format": "json"})
schema = r.json()
print(r.status_code, r["Content-Type"])
print(schema["info"])
for path, methods in schema["paths"].items():
    print(f"{path:24}", sorted(methods))
print("тіло POST:", schema["paths"]["/api/notes/"]["post"]["requestBody"]["content"]["application/json"]["schema"])

Чотири адреси, у кожної — дозволені методи (`put` немає, як у прикладі 2.8). Тіло `POST` посилається на компонент `NoteInput` — його drf-spectacular зібрав з `NoteInputSerializer`, бо в `api.py` над ViewSet стоїть `@extend_schema_view(...)` з `request=NoteInputSerializer`. Звичайний `ViewSet` сам не знає своїх серіалізаторів.

---
## 5. Той самий API на FastAPI

`fastapi_notes.py` у папці уроку — ті самі поля, обмеження й адреси, дані в словнику в пам'яті (база для FastAPI — урок 39). Валідація — Pydantic-моделі за анотаціями типів (урок 37):

```python
class NoteIn(BaseModel):
    title: str = Field(min_length=1, max_length=200)
    content: str = ""
    priority: int = Field(1, ge=1, le=4)
    is_pinned: bool = False
```

### Приклад 5.1. Створення і список

In [ ]:
from fastapi import FastAPI, HTTPException, Path, Query
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

import fastapi_notes

fastapi_notes.NOTES.clear()                            # дані в пам'яті — почнімо з чистого
fast = TestClient(fastapi_notes.app)
for body in ({"title": "Вивчити FastAPI", "priority": 4}, {"title": "Купити квитки", "priority": 2},
             {"title": "Прочитати про Pydantic"}):
    r = fast.post("/api/notes/", json=body)
    print(r.status_code, r.json())
print("список:", [n["title"] for n in fast.get("/api/notes/").json()])
print("пошук:", [n["title"] for n in fast.get("/api/notes/", params={"search": "купити"}).json()])

`content`, `is_pinned` і `id` додано за замовчуванням; `response_model=NoteOut` визначає, які поля піде у відповідь. Сортування — як у Django-версії: спершу закріплені, потім вищий пріоритет. `search` — звичайний параметр функції, FastAPI бере його з `?search=`.

### Приклад 5.2. Помилки: `422` у FastAPI, `400` у DRF

Ті самі неправильні дані — в обидві версії.

In [ ]:
bad = fast.post("/api/notes/", json={"title": "", "priority": 9})
fast_code = bad.status_code
fast_fields = {error["loc"][-1] for error in bad.json()["detail"]}

assert fast_code == 422 and fast_fields == {"title", "priority"}
api.force_authenticate(olena)
drf_bad = api.post("/api/notes/", {"title": "", "priority": 9}, format="json")
print("DRF:    ", drf_bad.status_code, json.dumps(drf_bad.json(), ensure_ascii=False))
print("FastAPI:", fast_code, [(e["loc"], e["msg"]) for e in bad.json()["detail"]])

Зміст однаковий — помилки за полями, — різні код і формат. DRF: словник «поле → список»; FastAPI: список `detail`, де `loc` — **шлях** до поля (`["body", "title"]`: у тілі запиту, поле `title`), бо параметр може прийти й з адреси чи заголовка. Тому поле — останній елемент `loc`.

### Приклад 5.3. `PATCH` з `exclude_unset`, `pin`, `DELETE`, `404`

In [ ]:
first_id = fast.get("/api/notes/").json()[0]["id"]
print("PATCH  →", fast.patch(f"/api/notes/{first_id}/", json={"title": "Вивчити FastAPI до п'ятниці"}).json())
print("pin    →", fast.post(f"/api/notes/{first_id}/pin/").json()["is_pinned"])
print("DELETE →", fast.delete(f"/api/notes/{first_id}/").status_code)
missing = fast.get(f"/api/notes/{first_id}/")
print("ще раз →", missing.status_code, missing.json())
print("не число →", fast.get("/api/notes/abc/").status_code)

- `PATCH` змінив лише `title`, `priority` лишився `4`: `body.model_dump(exclude_unset=True)` бере лише надіслані поля — це аналог `partial=True` у DRF;
- `HTTPException(404, ...)` → `{"detail": "..."}`;
- `/api/notes/abc/` — `422`, а не `404`: `note_id: int` в анотації, і `"abc"` не пройшов перевірку типу.

### Приклад 5.4. Схема — з коробки

In [ ]:
openapi = fastapi_notes.app.openapi()
print(openapi["info"])
for path, methods in openapi["paths"].items():
    print(f"{path:28}", sorted(methods))
print("NoteIn:", openapi["components"]["schemas"]["NoteIn"]["properties"]["priority"])

Жодного додаткового пакета: схему FastAPI зібрав з анотацій функцій і Pydantic-моделей. `ge=1, le=4` стали `minimum` / `maximum` — клієнт знає обмеження ще до запиту. Swagger UI для неї — `/docs` (якщо запустити `uvicorn fastapi_notes:app`).

### Приклад 5.5. Свій мінізастосунок: параметри шляху й запиту

Анотація параметра — це і є перевірка. `Path(ge=1, le=4)` обмежує число з адреси, `Query(min_length=2)` — параметр з `?q=`.

In [ ]:
PRIORITY_LABELS = {1: "🟢 Низький", 2: "🟡 Середній", 3: "🟠 Високий", 4: "🔴 Терміново"}
mini = FastAPI(title="Mini")


@mini.get("/api/priorities/{level}")
def priority_label(level: int = Path(ge=1, le=4)):
    return {"level": level, "label": PRIORITY_LABELS[level]}


@mini.get("/api/search/")
def search(q: str = Query(min_length=2), limit: int = 10):
    return {"q": q, "limit": limit}

In [ ]:
mini_client = TestClient(mini)
for url in ["/api/priorities/3", "/api/priorities/7", "/api/priorities/abc", "/api/search/?q=django&limit=5",
            "/api/search/?q=d", "/api/search/"]:
    r = mini_client.get(url)
    print(f"{url:32} {r.status_code}", r.json() if r.status_code == 200 else r.json()["detail"][0]["msg"])

Жодного `if` для перевірок: `7` — більше за `le=4`, `"abc"` — не число, `q=d` — коротше за 2, без `q` — параметр обов'язковий (немає значення за замовчуванням). Кожне — `422` з поясненням. `limit` без `?limit=` — `10`.

### Задача 3. API тегів на FastAPI

Допиши мінізастосунок `tags_app`:

- модель `TagIn`: `name` — рядок 1–30 символів, `color` — рядок за шаблоном `^#[0-9a-fA-F]{6}$` (`Field(..., pattern=...)`), за замовчуванням `"#808080"`;
- `create_tag(body: TagIn)`: ключ — `body.name.lower()`; якщо такий тег уже є в `TAGS` — `HTTPException(409, "Тег уже існує.")`; інакше збережи `body.model_dump()` у `TAGS[ключ]` і поверни його (статус `201` уже в декораторі).

Перевірка — у наступній клітинці.

In [ ]:
tags_app = FastAPI(title="Tags API")
TAGS: dict[str, dict] = {}


class TagIn(BaseModel):
    # YOUR CODE HERE
    pass


@tags_app.post("/api/tags/", status_code=201)
def create_tag(body: TagIn):
    # YOUR CODE HERE
    pass


In [ ]:
TAGS.clear()
tags = TestClient(tags_app)
created = tags.post("/api/tags/", json={"name": "django"})
assert created.json() == {"name": "django", "color": "#808080"}, f"новий тег з кольором за замовчуванням, а прийшло {created.json()}"
assert created.status_code == 201
assert tags.post("/api/tags/", json={"name": "FastAPI", "color": "#2E7D32"}).status_code == 201
duplicate = tags.post("/api/tags/", json={"name": "Django"})
assert duplicate.status_code == 409 and duplicate.json() == {"detail": "Тег уже існує."}, "той самий тег іншим регістром — 409"
assert tags.post("/api/tags/", json={"name": ""}).status_code == 422, "порожня назва — 422"
assert tags.post("/api/tags/", json={"name": "x" * 31}).status_code == 422, "назва довша за 30 — 422"
assert tags.post("/api/tags/", json={"name": "red", "color": "red"}).status_code == 422, "колір лише #RRGGBB"
assert sorted(TAGS) == ["django", "fastapi"]
print("✅ Задача 3 розв'язана")

---
## Самоперевірка

1. Навіщо два серіалізатори — вхідний і вихідний?
2. Що станеться з полем `user` у тілі `POST`? А з `fields = "__all__"`?
3. Чим `validate_<поле>()` відрізняється від `validate()`?
4. Чому `ViewSet`, а не `ModelViewSet`, у проєкті уроку?
5. Чим `PATCH` з `partial=True` відрізняється від `POST` щодо значень за замовчуванням? Що робить те саме у FastAPI?
6. Чому без входу `403`, а не `401`?
7. Де DRF повертає `400`, а FastAPI — `422`? Що спільного?

<details>
<summary>Відповіді</summary>

1. Клієнт бачить більше, ніж може змінити (`id`, `priority_label`, `updated_at`); вхідний приймає лише дозволене.
2. Його немає у вхідному серіалізаторі — ігнорується; власника задає `request.user`. З `"__all__"` поле `user` прийнялося б — нотатку можна було б створити від чужого імені.
3. `validate_title()` перевіряє одне поле й повертає його значення; `validate(attrs)` — кілька полів разом.
4. `ModelViewSet` сам робить `Note.objects.all()` і `save()` — обійшов би `services`/`selectors` і (як у прикладі 3.5) легко віддав би чужі дані.
5. `partial=True` не підставляє значень за замовчуванням: у `validated_data` лише надіслані поля — решта не змінюється. У FastAPI — `model_dump(exclude_unset=True)`.
6. `401` потребує заголовка `WWW-Authenticate`, а DRF бере його з першого класу автентифікації — сесійного, який такого заголовка не має.
7. Обидва — на помилки валідації вхідних даних з поясненням за полями; різні лише код і формат тіла (`{"поле": [...]}` проти `detail` з `loc`).

</details>

## Далі

- Книга курсу: [Урок 36](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_36/) — рефакторинг «+ API», діфи, архітектура, Browsable API, вибір між Django + DRF і FastAPI.
- Запусти проєкт: `python manage.py runserver` у папці `crispy_notes_project`, увійди на сайт і відкрий `http://127.0.0.1:8000/api/notes/`; тести — `python manage.py test`.
- **Урок 37** — типізація й Pydantic: перший крок новинного агрегатора.